# Two-session stop-signal ROI analysis with Boldtailor

This executable example follows two stop-signal runs from BIDS discovery through a bounded ROI fit, compact diagnostics, provenance projection, and transactional publication. It reports only shareable labels, basenames, array summaries, and derivative-relative paths.

In [ ]:
import os
from pathlib import Path

BIDS_ROOT = Path(
    os.environ.get("BOLDTAILOR_BIDS_ROOT", "/Users/poldrack/data_unsynced/rdoc_fmri")
)
FMRIPREP_ROOT = BIDS_ROOT / "derivatives" / "fmri_25.2.0"
SUBJECT = "sub-s4"
DEFAULT_SESSIONS = ("ses-06", "ses-08")


def _configured_sessions():
    selection = os.environ.get("BOLDTAILOR_SESSIONS")
    if selection is None:
        return DEFAULT_SESSIONS
    sessions = tuple(value.strip() for value in selection.split(","))
    if len(sessions) != 2 or not all(sessions):
        raise ValueError(
            "BOLDTAILOR_SESSIONS must select exactly two non-empty sessions"
        )
    return sessions


SESSIONS = _configured_sessions()
TASK = "stopSignal"
RUN = "run-01"
SPACE = "MNI152NLin2009cAsym"
RESOLUTION = 2
ROI_CENTER_MNI = (48.0, 16.0, 20.0)
ROI_RADIUS_MM = 6.0
PERSIST_DERIVATIVES = False
OVERWRITE_EXISTING = False
TEMPORARY_PARENT = os.environ.get("BOLDTAILOR_TEMP_ROOT")

## Imports and discovery

Boldtailor keeps package initializers empty, so this notebook imports each public module explicitly. Discovery resolves one raw events table and the matching preprocessed BOLD image, brain mask, and confounds table for each session. The display deliberately contains session labels and basenames only.

In [ ]:
import importlib
import matplotlib.pyplot as plt
import nilearn.plotting as plotting
import numpy as np
import pandas as pd
from IPython.display import display

import boldtailor.bids_provenance as bids_provenance
import boldtailor.data as data
import boldtailor.fit as fit
import boldtailor.model as model
import boldtailor.publication as publication


def _load_example_module():
    try:
        return importlib.import_module("examples.stop_signal_demo")
    except ModuleNotFoundError as error:
        if error.name != "examples":
            raise
        return importlib.import_module("stop_signal_demo")


_example = _load_example_module()
common_roi_voxels = _example.common_roi_voxels
discover_run_inputs = _example.discover_run_inputs
load_run = _example.load_run
protected_source_paths = _example.protected_source_paths
publication_destination = _example.publication_destination
result_artifacts = _example.result_artifacts
roi_image = _example.roi_image
run_sources = _example.run_sources

TRIAL_TYPES = ("go_success", "go_failure", "stop_success", "stop_failure")
CONFOUNDS = (
    "trans_x",
    "trans_y",
    "trans_z",
    "rot_x",
    "rot_y",
    "rot_z",
    "framewise_displacement",
)
inputs = tuple(
    discover_run_inputs(
        BIDS_ROOT,
        FMRIPREP_ROOT,
        subject=SUBJECT,
        session=session,
        task=TASK,
        run=RUN,
        space=SPACE,
        resolution=RESOLUTION,
    )
    for session in SESSIONS
)
display(
    pd.DataFrame(
        [
            {
                "session": item.session,
                "events": item.events.name,
                "bold": item.bold.name,
                "mask": item.mask.name,
                "confounds": item.confounds.name,
            }
            for item in inputs
        ]
    )
)

## Bounded ROI loading

The 6 mm sphere centered at MNI (48, 16, 20) is an **illustrative right inferior frontal ROI**, not an anatomical definition or inferential claim. Intersecting the sphere with both brain masks gives the same voxel ordering for both runs while loading only the smallest bounding blocks needed for that ROI.

In [ ]:
common_voxels = common_roi_voxels(
    inputs, center_mni=ROI_CENTER_MNI, radius_mm=ROI_RADIUS_MM
)
loaded_runs = tuple(
    load_run(
        item,
        common_voxels,
        trial_types=TRIAL_TYPES,
        confound_names=CONFOUNDS,
    )
    for item in inputs
)
run_summary = pd.DataFrame(
    [
        {
            "session": run.inputs.session,
            "signal_shape": tuple(run.signals.shape),
            "TR_seconds": run.tr,
            "scan_count": len(run.frame_times),
        }
        for run in loaded_runs
    ]
)
display(run_summary)
condition_counts = pd.DataFrame(
    {run.inputs.session: run.events.trial_type.value_counts() for run in loaded_runs}
).T.fillna(0).astype(int)
axes = condition_counts.reindex(columns=TRIAL_TYPES, fill_value=0).plot.bar(
    figsize=(9, 3), ylabel="event count", rot=0
)
axes.set_title("Modeled condition counts by session")
plt.tight_layout()
plt.show()

## Array ingestion

The loader hands time-by-voxel arrays and run-specific tables to Boldtailor. Dataset-relative source descriptors provide stable provenance without making the modeling API read or hash those files.

In [ ]:
shareable_configuration = {
    "subject": SUBJECT,
    "task": TASK,
    "sessions": list(SESSIONS),
    "roi": {
        "center_mni": list(ROI_CENTER_MNI),
        "radius_mm": ROI_RADIUS_MM,
    },
}
analysis_data = data.from_arrays(
    [run.signals for run in loaded_runs],
    [run.events for run in loaded_runs],
    frame_times=[run.frame_times for run in loaded_runs],
    confounds=[run.confounds for run in loaded_runs],
    sources=[run_sources(item, BIDS_ROOT) for item in inputs],
    provenance_metadata={
        "example": "two-session stop-signal ROI",
        "sessions": list(SESSIONS),
        "roi": {"center_mni": list(ROI_CENTER_MNI), "radius_mm": ROI_RADIUS_MM},
    },
)

## Run-specific designs and fit

Each run receives its own design matrix from its observed trial types rather than receiving fabricated empty regressors. The two requested contrasts remain estimable in both runs.

In [ ]:
model_spec = model.ModelSpec(
    contrasts={
        "successful_inhibition": "stop_success - stop_failure",
        "stop_vs_go": "stop_success - go_success",
    },
    confounds=CONFOUNDS,
    noise_model="ar1",
)
result = fit.fit(analysis_data, model_spec)
design_columns = tuple(tuple(frame.columns) for frame in result.design_matrices)
display(pd.DataFrame({"session": SESSIONS, "design_columns": design_columns}))

figure, design_axes = plt.subplots(1, 2, figsize=(15, 5))
for session, design, axis in zip(
    SESSIONS, result.design_matrices, design_axes, strict=True
):
    plotting.plot_design_matrix(design, axes=axis)
    axis.set_title(session)
figure.suptitle("Run-specific Boldtailor designs")
figure.tight_layout()
plt.show()

## Results and diagnostics

The result exposes all requested contrast quantities as ROI arrays. The compact tables below summarize effects, variances, test statistics, z-scores, directional one-sided p-values, per-run fit quality, aggregate fit quality, and the design construction metadata without dumping voxelwise arrays.

In [ ]:
contrast_rows = []
for contrast_name in result.contrast_names:
    contrast_rows.append(
        {
            "contrast": contrast_name,
            "mean_effect": float(np.mean(result.effect(contrast_name))),
            "mean_variance": float(np.mean(result.variance(contrast_name))),
            "mean_stat": float(np.mean(result.stat(contrast_name))),
            "mean_z_score": float(np.mean(result.z_score(contrast_name))),
            "min_one_sided_p_value": float(
                np.min(result.one_sided_p_value(contrast_name))
            ),
        }
    )
display(tuple(result.contrast_names))
display(pd.DataFrame(contrast_rows))
display(
    pd.DataFrame(
        [
            {"scope": session, "mean_r2": float(np.mean(values))}
            for session, values in zip(SESSIONS, result.run_r2, strict=True)
        ]
        + [{"scope": "aggregate", "mean_r2": float(np.mean(result.r2))}]
    )
)
display(pd.DataFrame(result.design_provenance, index=SESSIONS))
display({"warnings": [dict(item) for item in result.provenance.warnings]})
display(
    {
        "execution_id": result.provenance.execution_id,
        "analysis_fingerprint": result.provenance.analysis_fingerprint,
    }
)
successful_inhibition_image = roi_image(
    result.z_score("successful_inhibition"), loaded_runs[0]
)
plotting.plot_stat_map(
    successful_inhibition_image,
    cut_coords=ROI_CENTER_MNI,
    title="Illustrative ROI: successful inhibition z-score",
)
plt.show()

## Structured logging and provenance

The bounded provenance history mirrors selected structured lifecycle fields rather than raw data or the full provenance record. A fresh execution ID correlates events from this fit attempt; the analysis fingerprint instead identifies the reproducible combination of complete source metadata and model specification.

In [ ]:
event_fields = (
    "sequence",
    "stage",
    "event",
    "level",
    "execution_id",
    "analysis_id",
)
selected_events = pd.DataFrame(
    [
        {field: event.get(field) for field in event_fields}
        for event in result.provenance.events
    ]
)
display(selected_events)

## BIDS provenance projection

The canonical record is projected in memory to stable BIDS derivative metadata plus Boldtailor's pinned BEP028 draft subset. Only derivative-relative paths and payload byte sizes are shown.

In [ ]:
projected = bids_provenance.project_bids_provenance(
    result.provenance,
    dataset_name="Boldtailor stop-signal example derivatives",
    label="boldtailor",
)
display(
    pd.DataFrame(
        [
            {"relative_path": path, "byte_size": len(payload)}
            for path, payload in projected.items()
        ]
    )
)

## Transactional publication

Projection bytes and compact reports form one artifact set that is validated, staged, synced, and published transactionally. The safe default creates a temporary `boldtailor-*` directory. Setting `PERSIST_DERIVATIVES=True` writes only to `<dataset>/derivatives/boldtailor`; overwrite remains an explicit, separate choice.

In [ ]:
artifacts = tuple(
    publication.Artifact(path, payload) for path, payload in projected.items()
)
artifacts += result_artifacts(
    result,
    subject=SUBJECT,
    task=TASK,
    sessions=SESSIONS,
    configuration=shareable_configuration,
)
destination = publication_destination(
    BIDS_ROOT,
    persistent=PERSIST_DERIVATIVES,
    temporary_parent=TEMPORARY_PARENT,
)
published = publication.publish_artifact_set(
    destination,
    artifacts,
    source_paths=protected_source_paths(inputs),
    overwrite=OVERWRITE_EXISTING,
)
display(
    {
        "destination_basename": destination.name,
        "artifact_count": len(artifacts),
        "published_count": len(published),
    }
)
display(
    pd.DataFrame(
        {
            "artifact_relative_path": [
                path.relative_to(destination).as_posix() for path in published
            ]
        }
    )
)